# 검증 실험 (Colab)

블로그·README의 근거 없는 설명을 실측으로 바꾸기 위한 노트북. **런타임 > 런타임 유형 변경 > T4 GPU** 후 **모두 실행**.

In [ ]:
!nproc; lscpu | grep -E 'Model name|^CPU\(s\)|Thread|Core|Socket|MHz'; python -V; nvidia-smi --query-gpu=name,memory.total,clocks.max.mem --format=csv

## 1. GIL: 순차 / 스레드 / 프로세스

In [ ]:
%%writefile gil_vs_multiprocess.py
"""
LangGraph의 BackgroundExecutor가 실제로 ThreadPoolExecutor를 쓴다는 걸 소스에서 확인했다
(langgraph/pregel/_executor.py: "Uses a thread pool executor to delegate tasks to
separate threads."). 그렇다면 노드 함수가 CPU-bound일 때 "병렬 실행"이 진짜 멀티코어를
쓰는지, 아니면 GIL 때문에 동시성일 뿐인지 직접 측정한다.

비교 대상:
  1. 순차 실행 (baseline)
  2. ThreadPoolExecutor (LangGraph가 실제로 쓰는 것과 동일한 메커니즘)
  3. ProcessPoolExecutor (진짜 멀티코어 병렬)

워크로드: 순수 CPU-bound(요청 검증/스키마 체크 같은 걸 흉내낸 합성 연산) vs
I/O-bound(LLM API 호출을 흉내낸 time.sleep, GIL이 풀리는 경우)
"""

import time
import os
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor

N_TASKS = 8  # os.cpu_count()와 맞춤


def cpu_bound_task(n: int) -> int:
    """순수 CPU 연산 -- LangGraph 노드가 파이썬 로직만 돌리는 경우를 흉내냄."""
    total = 0
    for i in range(20_000_000):
        total += i * i % 7
    return total


def io_bound_task(n: int) -> float:
    """LLM API 호출처럼 네트워크 대기가 있는 경우를 흉내냄 (GIL 풀림)."""
    time.sleep(0.3)
    return n


def run_sequential(fn, n_tasks):
    t0 = time.perf_counter()
    for i in range(n_tasks):
        fn(i)
    return time.perf_counter() - t0


def run_threadpool(fn, n_tasks):
    t0 = time.perf_counter()
    with ThreadPoolExecutor(max_workers=n_tasks) as ex:
        list(ex.map(fn, range(n_tasks)))
    return time.perf_counter() - t0


def run_processpool(fn, n_tasks):
    t0 = time.perf_counter()
    with ProcessPoolExecutor(max_workers=n_tasks) as ex:
        list(ex.map(fn, range(n_tasks)))
    return time.perf_counter() - t0


if __name__ == "__main__":
    print(f"CPU 코어 수: {os.cpu_count()}, 태스크 수: {N_TASKS}\n")

    print("=== 1. CPU-bound 워크로드 (LangGraph 노드가 순수 파이썬 로직을 돌리는 경우) ===")
    t_seq = run_sequential(cpu_bound_task, N_TASKS)
    t_thread = run_threadpool(cpu_bound_task, N_TASKS)
    t_proc = run_processpool(cpu_bound_task, N_TASKS)
    print(f"순차 실행:            {t_seq:6.2f}s")
    print(f"ThreadPoolExecutor:   {t_thread:6.2f}s  (순차 대비 {t_seq/t_thread:.2f}x)  <- LangGraph가 쓰는 것과 동일한 메커니즘")
    print(f"ProcessPoolExecutor:  {t_proc:6.2f}s  (순차 대비 {t_seq/t_proc:.2f}x)  <- 진짜 멀티코어 병렬")

    print("\n=== 2. I/O-bound 워크로드 (LLM API 호출처럼 대기가 있는 경우) ===")
    t_seq2 = run_sequential(io_bound_task, N_TASKS)
    t_thread2 = run_threadpool(io_bound_task, N_TASKS)
    print(f"순차 실행:            {t_seq2:6.2f}s")
    print(f"ThreadPoolExecutor:   {t_thread2:6.2f}s  (순차 대비 {t_seq2/t_thread2:.2f}x)  <- 여기서는 스레드가 잘 먹힘(GIL이 sleep 동안 풀림)")


In [ ]:
%%writefile gil_extra.py
import sys, time, os
from concurrent.futures import ProcessPoolExecutor
from gil_vs_multiprocess import cpu_bound_task, io_bound_task, run_sequential, run_threadpool, run_processpool
def noop(i): return i
def pool(w, fn, n=8):
    t0=time.perf_counter()
    with ProcessPoolExecutor(max_workers=w) as ex: list(ex.map(fn, range(n)))
    return time.perf_counter()-t0
if __name__=='__main__':
    N=os.cpu_count(); print('cpu_count', N)
    for r in range(3):
        s=run_sequential(cpu_bound_task,N); t=run_threadpool(cpu_bound_task,N); p=run_processpool(cpu_bound_task,N)
        print(f'run{r} order=seq,thr,proc tasks={N}: seq {s:.2f}s thr {s/t:.3f}x proc {s/p:.3f}x')
    for r in range(2):
        t=run_threadpool(cpu_bound_task,N); s=run_sequential(cpu_bound_task,N)
        print(f'swap{r} order=thr,seq: thr {s/t:.3f}x')
    t0=time.perf_counter(); cpu_bound_task(0); one=time.perf_counter()-t0
    print(f'single task {one:.2f}s; pool startup+noop({N} workers) {pool(N, noop, N):.3f}s')
    for w in sorted({1,2,max(1,N//2),N}):
        t=pool(w, cpu_bound_task, N); print(f'{w} workers, {N} tasks: {N*one/t:.2f}x (ideal {min(w,N)}x)')
    s=run_sequential(io_bound_task,N); t=run_threadpool(io_bound_task,N); print(f'io: thr {s/t:.2f}x (ideal {N}x)')


In [ ]:
!python gil_extra.py

## 2. Triton group-quant 커널 (T4)

In [ ]:
%%writefile group_quant_triton.py
"""Per-group INT8 quantization kernel written in Triton.

Simplified re-implementation of the algorithm in vLLM's
csrc/libtorch_stable/quantization/w8a8/fp8/per_token_group_quant.cu
(per_token_group_quant_8bit_kernel), but using Triton's DSL instead of
raw CUDA. One Triton "program" (= one block) handles exactly one group.

For each group of `group_size` contiguous elements:
  1. load the group
  2. scale = max(|x|) / 127          (absmax quantization)
  3. store round(x / scale) as int8, and store `scale` separately
"""

import torch
import triton
import triton.language as tl


@triton.jit
def _group_quant_int8_kernel(
    x_ptr,          # *input, shape [num_tokens, hidden_size], row-major
    out_q_ptr,      # *output int8, same shape as x
    out_s_ptr,      # *output fp32 scales, shape [num_tokens, num_groups]
    hidden_size,    # int
    group_size: tl.constexpr,
):
    # One program == one (token, group) pair.
    pid = tl.program_id(axis=0)
    num_groups = hidden_size // group_size
    token_id = pid // num_groups
    group_id = pid % num_groups

    offs = tl.arange(0, group_size)
    row_start = token_id * hidden_size + group_id * group_size

    x = tl.load(x_ptr + row_start + offs).to(tl.float32)

    absmax = tl.max(tl.abs(x), axis=0)
    scale = absmax / 127.0
    scale = tl.where(scale == 0.0, 1.0, scale)  # avoid div-by-zero on all-zero groups

    # vLLM's reference kernel does a plain truncating cast here (no
    # rounding) - `dst = DST_DTYPE(q)` in per_token_group_quant.cu - so we
    # match that instead of rounding to nearest.
    q = tl.minimum(tl.maximum(x / scale, -127.0), 127.0)

    tl.store(out_q_ptr + row_start + offs, q.to(tl.int8))
    tl.store(out_s_ptr + token_id * num_groups + group_id, scale)


def group_quant_int8_triton(
    x: torch.Tensor, group_size: int
) -> tuple[torch.Tensor, torch.Tensor]:
    """Quantize the last dim of `x` in chunks of `group_size` to int8.

    Args:
        x: [num_tokens, hidden_size] floating point tensor on CUDA.
        group_size: must evenly divide hidden_size.

    Returns:
        (q, scales): q is int8 [num_tokens, hidden_size],
        scales is fp32 [num_tokens, hidden_size // group_size].
    """
    assert x.is_cuda, "Triton kernels need a CUDA tensor (use a Colab GPU runtime)"
    assert x.ndim == 2
    num_tokens, hidden_size = x.shape
    assert hidden_size % group_size == 0

    num_groups = hidden_size // group_size
    out_q = torch.empty_like(x, dtype=torch.int8)
    out_s = torch.empty((num_tokens, num_groups), dtype=torch.float32, device=x.device)

    grid = (num_tokens * num_groups,)
    _group_quant_int8_kernel[grid](
        x, out_q, out_s, hidden_size, group_size=group_size
    )
    return out_q, out_s


In [ ]:
%%writefile group_quant_triton_multi.py
"""Multi-group-per-program variant of the Triton group-quant kernel.

The original kernel (group_quant_triton.py) launches one program per group.
vLLM's CUDA kernel instead lets one block handle several groups
(`groups_per_block`). This variant does the same in Triton: each program
loads a [GROUPS, group_size] tile, reduces along the group axis and writes
GROUPS scales.

Because x is row-major and groups are contiguous along hidden_size, group g
(counted across all tokens) starts at element g * group_size and its scale
lives at out_s[g] when out_s is [num_tokens, num_groups] row-major.
"""

import torch
import triton
import triton.language as tl


@triton.jit
def _group_quant_int8_multi_kernel(
    x_ptr,
    out_q_ptr,
    out_s_ptr,
    total_groups,
    group_size: tl.constexpr,
    GROUPS: tl.constexpr,
):
    pid = tl.program_id(axis=0)
    g = pid * GROUPS + tl.arange(0, GROUPS)
    g_mask = g < total_groups
    offs = g[:, None] * group_size + tl.arange(0, group_size)[None, :]

    x = tl.load(x_ptr + offs, mask=g_mask[:, None], other=0.0).to(tl.float32)

    absmax = tl.max(tl.abs(x), axis=1)
    scale = absmax / 127.0
    scale = tl.where(scale == 0.0, 1.0, scale)

    q = tl.minimum(tl.maximum(x / scale[:, None], -127.0), 127.0)

    tl.store(out_q_ptr + offs, q.to(tl.int8), mask=g_mask[:, None])
    tl.store(out_s_ptr + g, scale, mask=g_mask)


def group_quant_int8_triton_multi(
    x: torch.Tensor, group_size: int, groups_per_program: int = 8
) -> tuple[torch.Tensor, torch.Tensor]:
    """Same contract as group_quant_int8_triton, GROUPS groups per program."""
    assert x.is_cuda and x.ndim == 2 and x.is_contiguous()
    num_tokens, hidden_size = x.shape
    assert hidden_size % group_size == 0
    # tl.arange needs power-of-two extents
    assert groups_per_program & (groups_per_program - 1) == 0

    num_groups = hidden_size // group_size
    total_groups = num_tokens * num_groups
    out_q = torch.empty_like(x, dtype=torch.int8)
    out_s = torch.empty((num_tokens, num_groups), dtype=torch.float32, device=x.device)

    grid = (triton.cdiv(total_groups, groups_per_program),)
    _group_quant_int8_multi_kernel[grid](
        x, out_q, out_s, total_groups,
        group_size=group_size, GROUPS=groups_per_program,
    )
    return out_q, out_s


In [ ]:
import torch, triton, triton.language as tl, statistics, json
from group_quant_triton import _group_quant_int8_kernel, group_quant_int8_triton
from group_quant_triton_multi import group_quant_int8_triton_multi
print(torch.__version__, triton.__version__, torch.cuda.get_device_name())
H, G = 4096, 128
def ref(x, g):
    n,h=x.shape; xg=x.reshape(n,h//g,g).float(); s=(xg.abs().amax(-1)/127.0).clamp_min(1e-12)
    return (xg/s.unsqueeze(-1)).clamp(-127,127).to(torch.int8).reshape(n,h), s
def tri1(x, g, nw):
    n,h=x.shape; q=torch.empty_like(x,dtype=torch.int8); s=torch.empty((n,h//g),dtype=torch.float32,device=x.device)
    _group_quant_int8_kernel[(n*(h//g),)](x,q,s,h,group_size=g,num_warps=nw); return q,s
def bench(fn):
    return statistics.median(triton.testing.do_bench(fn, return_mode='median') for _ in range(5))*1000
def gbps(n,us): return (n*H*2 + n*H + n*(H//G)*4)/(us*1e-6)/1e9
x=torch.empty(256*1024*1024//2, dtype=torch.float16, device='cuda'); y=torch.empty_like(x)
cp=bench(lambda: y.copy_(x)); print(f'copy 512MiB: {2*x.numel()*2/(cp*1e-6)/1e9:.1f} GB/s (read+write)'); del x,y
rows=[]
for n in (16,256,4096):
    x=torch.randn(n,H,device='cuda',dtype=torch.float16)*5
    r={'tokens':n,'pytorch_ref':bench(lambda: ref(x,G))}
    for nw in (1,2,4): r[f'tri1_w{nw}']=bench(lambda: tri1(x,G,nw))
    for k in (1,2,4,8,16): r[f'multi{k}']=bench(lambda: group_quant_int8_triton_multi(x,G,k))
    rows.append(r)
    print(n, ' '.join(f'{k}={v:.1f}us' for k,v in r.items() if k!='tokens'))
print('4096-token effective GB/s:', {k: round(gbps(4096,v),1) for k,v in rows[-1].items() if k!='tokens'})
json.dump(rows, open('bench_t4.json','w'), indent=1)

In [ ]:
torch.manual_seed(0)
x=torch.randn(4096,H,device='cuda',dtype=torch.float16)*5
q1,s1=group_quant_int8_triton(x,G); qm,sm=group_quant_int8_triton_multi(x,G,8); qr,sr=ref(x,G)
d=(q1.int()-qr.int()).abs(); print('triton vs ref: mismatches', int((d!=0).sum()), 'of', d.numel(), 'max', int(d.max()), 'scale equal', torch.equal(s1,sr))
print('multi8 == single bitwise:', torch.equal(q1,qm) and torch.equal(s1,sm))
raw=(x.float().reshape(4096,-1,G)/s1.unsqueeze(-1)).reshape(4096,H); idx=d.nonzero()[:5]
for i,j in idx.tolist(): print(i,j,'x/scale(torch)=%.6f'%raw[i,j].item(),'q_triton',q1[i,j].item(),'q_ref',qr[i,j].item())